# Tests: `fasterai.sparse.sparsifier` (source `nbs/sparse/sparsifier.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.core.criteria import activation_criteria, large_final
from fasterai.sparse.sparsifier import *

In [ ]:
from fastcore.test import *
import warnings

def _test_model():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10)
    )

def _zeros_pct(model):
    z = sum((m.weight==0).sum().item() for m in model.modules() if isinstance(m, nn.Conv2d))
    t = sum(m.weight.numel() for m in model.modules() if isinstance(m, nn.Conv2d))
    return 100 * z / t

# A fraction means what it says: 0.5 zeroes half the weights
conv = nn.Conv2d(3, 16, 3)
sp = Sparsifier(nn.Sequential(conv), 'weight', 'local', large_final, layer_type=nn.Conv2d)
sp.sparsify_layer(conv, 0.5)
test_close((conv.weight == 0).float().mean().item() * 100, 50.0, eps=5.0)

# Buffers created
assert hasattr(conv, '_mask')
assert hasattr(conv, '_init_weights')

# Clean buffers
sp._clean_buffers()
assert not hasattr(conv, '_mask')

# sparsify_model with a fraction
model = _test_model()
Sparsifier(model, 'weight', 'local', large_final).sparsify_model(0.3)
test_close(_zeros_pct(model), 30.0, eps=5.0)

# A percent is read as x/100 for one release, and warns
_pmodel = _test_model()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    Sparsifier(_pmodel, 'weight', 'local', large_final).sparsify_model(50)
test_close(_zeros_pct(_pmodel), 50.0, eps=5.0)
test_eq({x.category for x in w}, {FutureWarning})
assert 'looks like a percent' in str(w[0].message)

# A fraction never warns
_fmodel = _test_model()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    Sparsifier(_fmodel, 'weight', 'local', large_final).sparsify_model(0.5)
test_eq(len(w), 0)

# Dict + context='global' raises ValueError
model2 = _test_model()
sp_g = Sparsifier(model2, 'weight', 'global', large_final)
with ExceptionExpected(ValueError):
    sp_g.sparsify_model({'0': 0.3, '3': 0.6})

# Per-layer dict: each layer reaches its own fraction, and a bad value names its layer
model_d = _test_model()
Sparsifier(model_d, 'weight', 'local', large_final).sparsify_model({'0': 0.3, '3': 0.6})
test_close((model_d[0].weight==0).float().mean().item(), 0.3, eps=0.05)
test_close((model_d[3].weight==0).float().mean().item(), 0.6, eps=0.05)
with ExceptionExpected(ValueError, regex="'3'"):
    Sparsifier(_test_model(), 'weight', 'local', large_final).sparsify_model({'0': 0.3, '3': 150})

# Invalid sparsity
model3 = _test_model()
sp3 = Sparsifier(model3, 'weight', 'local', large_final)
conv3 = nn.Conv2d(3, 16, 3)
with ExceptionExpected(ValueError): sp3.sparsify_layer(conv3, 150)
with ExceptionExpected(ValueError): sp3.sparsify_layer(conv3, -0.1)
with ExceptionExpected(TypeError): sp3.sparsify_layer(conv3, True)
with ExceptionExpected(TypeError): sp3.sparsify_model('0.4')

# print_sparsity runs without error
model4 = _test_model()
sp4 = Sparsifier(model4, 'weight', 'local', large_final)
sp4.sparsify_model(0.5)
sp4.print_sparsity()

# save_model writes a buffer-free model (regression: `copy` used to be missing at import time)
import tempfile, os
model5 = _test_model()
sp5 = Sparsifier(model5, 'weight', 'local', large_final)
sp5.sparsify_model(0.5)
with tempfile.TemporaryDirectory() as _d:
    _path = os.path.join(_d, 'ticket.pth')
    sp5.save_model(_path)
    assert os.path.exists(_path)
    _reloaded = torch.load(_path, weights_only=False)
for m in _reloaded.modules():
    assert not hasattr(m, '_mask') and not hasattr(m, '_init_weights')

# --- Wanda with Sparsifier ---
_wdata = [torch.randn(4, 3, 8, 8)]

# Wanda without data raises ValueError
with ExceptionExpected(ValueError):
    Sparsifier(_test_model(), 'weight', 'local', activation_criteria(torch.abs))

# Wanda with data works
_wmodel2 = _test_model()
sp_w = Sparsifier(_wmodel2, 'weight', 'local', activation_criteria(torch.abs), data=_wdata)
sp_w.sparsify_model(0.5)
test_close(_zeros_pct(_wmodel2), 50.0, eps=5.0)

In [ ]:
#| slow
from torchvision.models import resnet18
model_lg = resnet18(weights=None)
sp_lg = Sparsifier(model_lg, 'weight', 'local', large_final)
sp_lg.sparsify_model(0.6)
total_zeros = sum((m.weight==0).sum().item() for m in model_lg.modules() if isinstance(m, nn.Conv2d))
total_params = sum(m.weight.numel() for m in model_lg.modules() if isinstance(m, nn.Conv2d))
test_close(100*total_zeros/total_params, 60.0, eps=10.0)